**installations of libraries.**

In [1]:
pip install openpyxl


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install requests


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install mysql-connector-python


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pymysql


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**necessary libraries.**

In [34]:
import pandas as pd
import pymysql
import numpy as np


**loading the data**

In [40]:

file_path = r"C:\Users\Fame\Desktop\Thunder - Softoo\ThunderSSUData.xlsx"
df = pd.read_excel(file_path)

df = df.rename(columns={
    'Timestamp': 'ts',  # Example, rename accordingly
    'ac_dig_ac_source': 'ac_dig_ac_source',
    'main_contrib_batt_kwh': 'main_contrib_batt_kwh',
    'solar_contrib_batt_kwh': 'solar_contrib_batt_kwh',
    'dg_contrib_batt_kwh': 'dg_contrib_batt_kwh'
})

df = df.replace({np.nan: None})

df['ts'] = pd.to_datetime(df['ts'], format='mixed', utc=True) \
             .dt.tz_convert(None) \
             .dt.floor('S')


C:\Users\Fame\AppData\Local\Temp\ipykernel_8664\4199864539.py:16: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  .dt.floor('S')


In [41]:
df

,sitecode,ts,ac_dig_ac_source,source_tag,ls_status,main_contrib_batt_kwh,dg_contrib_batt_kwh,solar_contrib_batt_kwh
0,ABCD01,2025-01-19 07:50:28,1.0,Main,False,None,None,None
1,ABCD01,2025-01-19 07:55:28,1.0,Main,False,None,None,None
2,ABCD01,2025-01-19 08:00:27,1.0,Main,False,None,None,None
3,ABCD01,2025-01-19 08:05:28,1.0,Main,False,None,None,None
4,ABCD01,2025-01-19 08:10:28,1.0,Main,False,None,None,None
...,...,...,...,...,...,...,...,...
8614,ABCD01,2025-02-18 07:25:25,0.0,Solar+Battery,True,None,None,None
8615,ABCD01,2025-02-18 07:30:25,0.0,Solar+Battery,True,None,None,None
8616,ABCD01,2025-02-18 07:35:27,0.0,Solar+Battery,True,None,None,None
8617,ABCD01,2025-02-18 07:40:26,0.0,Solar+Battery,True,None,None,None


In [48]:
import pandas as pd
import mysql.connector
from datetime import datetime

# Step 1: Read the Excel file
file_path = r"C:\Users\Fame\Desktop\Thunder - Softoo\data.xlsx"
df = pd.read_excel(file_path)

# Step 2: Replace NaN/NULL values with 0
df.fillna(0, inplace=True)

# Step 3: Remove the "+05" part from the 'ts' column
def clean_timestamp(ts):
    if isinstance(ts, str):  # Ensure the value is a string
        return ts.split('+')[0]  # Remove the "+05" part
    return ts  # Return as-is if not a string

df['ts'] = df['ts'].apply(clean_timestamp)

# Step 4: Convert 'ts' column to datetime format
try:
    # Try parsing with fractional seconds first
    df['ts'] = pd.to_datetime(df['ts'], format='%Y-%m-%d %H:%M:%S.%f', errors='coerce')
except Exception:
    # If parsing fails, try without fractional seconds
    df['ts'] = pd.to_datetime(df['ts'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# Ensure no NaT values remain (replace them with a default timestamp if necessary)
df['ts'] = df['ts'].fillna(pd.Timestamp('1970-01-01'))

# Step 5: Ensure 'ac_dig_ac_source' is an integer
df['ac_dig_ac_source'] = df['ac_dig_ac_source'].astype(int)

# Step 6: Convert 'ls_status' from True/False to 1/0
df['ls_status'] = df['ls_status'].astype(int)  # Converts True to 1 and False to 0

# Step 7: Connect to the MySQL database
try:
    connection = mysql.connector.connect(
        host="localhost",          # Replace with your host (e.g., localhost)
        user="root",               # Replace with your username
        password="",               # Replace with your password
        database="thunder telenor site analysis"  # Replace with your database name
    )
    cursor = connection.cursor()

    # Step 8: Insert data into the database
    insert_query = """
    INSERT INTO thunder_data (
        sitecode, ts, ac_dig_ac_source, csu_ana_batt_total_curr, source_tag, 
        ls_status, mainkw, solarkw, dgkw, batterykw, main_contrib_batt_kwh, 
        dg_contrib_batt_kwh, solar_contrib_batt_kwh
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """

    for _, row in df.iterrows():
        cursor.execute(insert_query, (
            row['sitecode'],
            row['ts'],
            row['ac_dig_ac_source'],
            row['csu_ana_batt_total_curr'],
            row['source_tag'],
            row['ls_status'],  # Already converted to 1 or 0
            row['mainkw'],
            row['solarkw'],
            row['dgkw'],
            row['batterykw'],
            row['main_contrib_batt_kwh'],
            row['dg_contrib_batt_kwh'],
            row['solar_contrib_batt_kwh']
        ))

    # Commit the transaction
    connection.commit()
    print("Data inserted successfully!")

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # Close the database connection
    if connection.is_connected():
        cursor.close()
        connection.close()

Data inserted successfully!


In [49]:
print(ls_status)

NameError: name 'ls_status' is not defined

**in the dataset i was provided there were some attributes that were not in alignment with my project so i dropped those columns and only kept the ones i need.**

**okay so basically, before i tried to enter the dataset into the database which i created on the phpmyadmin.**

**it didnt accept my data because it contained NaN values.**

**so what im going to do is that i will first find out how many NaN values i have in my data set and, i will change those NaN to 0.**

**data manipulation is all done.**

**creating connection with the database and inserting the data.**

In [3]:
import plotly.graph_objects as go
import pymysql
from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

app = Flask(__name__)
CORS(app)

DB_CONFIG = {
        "host": "localhost",
        "user": "root",
        "password": "",
        "database": "thunder telenor site analysis"
    }

def get_db_connection():
        """Creates and returns a database connection."""
        return pymysql.connect(**DB_CONFIG, cursorclass=pymysql.cursors.DictCursor)

def fetch_power_data(timestamp):
        """Fetch power source data from the database for a given timestamp."""
        try:
            conn = get_db_connection()
            cursor = conn.cursor()
            cursor.execute("SELECT mainkw, solarkw, dgkw, batterykw FROM sitedata WHERE ts = %s", (timestamp,))
            result = cursor.fetchone()
            conn.close()
            return result if result else None
        except pymysql.MySQLError as e:
            print("Database Error:", e)
            return None


@app.route('/fetch-chart', methods=['GET'])
def fetch_chart():
        """API to return a JSON response with Plotly graph data for the given timestamp."""
        timestamp = request.args.get('ts')
        if not timestamp:
            return jsonify({"error": "Timestamp is required"}), 400

        data = fetch_power_data(timestamp)

        if data:
            # Ensure None values are replaced with 0
            mainkw = data.get('mainkw', 0) or 0
            solarkw = data.get('solarkw', 0) or 0
            dgkw = data.get('dgkw', 0) or 0
            batterykw = data.get('batterykw', 0) or 0

            sources = ['Main', 'Solar', 'DG', 'Battery']
            values = [mainkw, solarkw, dgkw, batterykw]

            # Creating Plotly figure
            fig = go.Figure()
            fig.add_trace(go.Bar(
                x=sources,
                y=values,
                marker_color=['blue', 'yellow', 'red', 'green']
            ))

            fig.update_layout(
                title=f"Power Source Distribution for {timestamp}",
                xaxis_title="Power Sources",
                yaxis_title="Power (kW)",
                plot_bgcolor="#161b22",
                paper_bgcolor="#161b22",
                font=dict(color="white")
            )

            # Convert to JSON format for frontend
            graph_json = fig.to_json()

            return jsonify({
                "graph": graph_json,
                "mainkw": mainkw,
                "solarkw": solarkw,
                "dgkw": dgkw,
                "batterykw": batterykw
            })

        else:
            return jsonify({"error": "No data found for the provided timestamp"}), 404


@app.route('/get-timestamps', methods=['GET'])
def get_timestamps():
        """Fetch the last 100 timestamps from the database."""
        try:
            conn = get_db_connection()
            cursor = conn.cursor()
            cursor.execute("SELECT ts FROM sitedata ORDER BY ts DESC LIMIT 100")
            timestamps = [row['ts'].strftime('%Y-%m-%d %H:%M:%S') for row in cursor.fetchall()]
            conn.close()
            return jsonify(timestamps)
        except pymysql.MySQLError as e:
            print("Database Error:", e)
            return jsonify({"error": "Failed to fetch timestamps"}), 500


@app.route('/login', methods=['POST'])
def login():
        """Handles user login."""
        data = request.get_json()
        username = data.get("username")
        password = data.get("password")

        try:
            conn = get_db_connection()
            cursor = conn.cursor()

            cursor.execute("SELECT id FROM users WHERE username = %s AND password = %s", (username, password))
            user = cursor.fetchone()
            conn.close()

            if user:
                return jsonify({"success": True, "message": "Login successful", "user_id": user["id"]})
            return jsonify({"success": False, "message": "Invalid credentials"}), 401

        except pymysql.MySQLError as e:
            print("Database Error:", e)
            return jsonify({"success": False, "message": "Database error"}), 500


@app.route('/get-load-share', methods=['GET'])
def get_load_share():
        """Calculate load share percentage for a given timestamp."""
        timestamp = request.args.get('ts')

        if not timestamp:
            return jsonify({"error": "Missing timestamp"}), 400

        conn = get_db_connection()
        cursor = conn.cursor()

        query = "SELECT mainkw, solarkw, dgkw, batterykw FROM sitedata WHERE ts = %s"
        cursor.execute(query, (timestamp,))
        result = cursor.fetchone()
        conn.close()

        if not result:
            return jsonify({"error": "No data found for the selected timestamp"}), 404

        mainkw, solarkw, dgkw, batterykw = result.values()
        total_power = mainkw + solarkw + dgkw + batterykw

        if total_power == 0:
            return jsonify({"error": "Total power is zero, cannot calculate percentage"}), 400

        load_share = {
            "Main Grid": round((mainkw / total_power) * 100, 2),
            "Solar": round((solarkw / total_power) * 100, 2),
            "Diesel Generator": round((dgkw / total_power) * 100, 2),
            "Battery": round((batterykw / total_power) * 100, 2)
        }

        return jsonify(load_share)


def fetch_loadshedding_data():
    """Fetches load shedding data from MySQL."""
    conn = get_db_connection()
    cursor = conn.cursor()
    query = "SELECT ts, ls_status FROM sitedata ORDER BY ts ASC"
    cursor.execute(query)
    data = cursor.fetchall()
    conn.close()

    # Ensure both columns are assigned correctly
    df = pd.DataFrame(data, columns=['ts', 'ls_status'])
    df['ts'] = pd.to_datetime(df['ts'])
    df = df.sort_values('ts')

    return df


@app.route('/predict', methods=['GET'])
def predict_loadshedding():
    """Uses SARIMA to forecast load shedding for the next 7 days."""
    data = fetch_loadshedding_data()

    # Ensure data is correctly formatted
    data.set_index('ts', inplace=True)

    # Resample to weekly averages
    weekly_avg = data['ls_status'].resample('W').mean()

    # SARIMA Model
    model = SARIMAX(
        weekly_avg, 
        order=(1, 1, 1), 
        seasonal_order=(1, 1, 1, 7), 
        enforce_stationarity=False, 
        enforce_invertibility=False
    )
    sarima_model = model.fit(disp=False)

    # Forecast Next 7 Days
    forecast_steps = 7  # Changed from 10 to 7 days
    forecast = sarima_model.get_forecast(steps=forecast_steps)
    forecast_mean = forecast.predicted_mean

    # Generate forecasted dates
    forecast_dates = pd.date_range(start=data.index.max(), periods=forecast_steps+1, freq='D')[1:]

    # Return JSON format
    return jsonify([{"date": str(forecast_dates[i]), "value": round(forecast_mean[i], 2)} for i in range(forecast_steps)])

data = fetch_loadshedding_data()
print(data['ls_status'].value_counts(normalize=True))  # See distribution of 0s and 1s



if __name__ == '__main__':
        app.run(host="0.0.0.0", port=5000, debug=True)


ls_status
0    0.909144
1    0.090856
Name: proportion, dtype: float64
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.26.5.67:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\Fame\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
